## 04 · LangChain Deep Agents ile belge ajanı (Colab · T4 GPU)

[Deep Agents](https://github.com/langchain-ai/deepagents) çerçevesiyle, 03 not defterindeki getirme hattını (**E5 + BM25 hibrit + bge-reranker**) araç olarak kullanan bir ajan kuruyorum.

- **Model:** `qwen3:4b-instruct-2507` (Ollama, yerel; veri dışarı çıkmaz)
- **Planlama:** `write_todos` (TodoListMiddleware), çok konulu soruyu adımlara böler
- **Araç:** `belge_ara`, reranker skoru eşiğin altındaysa *bulunamadı* döndürür

> Çalışma zamanı: *Runtime → Change runtime type → T4 GPU*

In [ ]:
# Ortam hazırlığı: Colab'da repo klonlanır, yerelde notebooks/ klasöründen proje köküne geçilir
import os, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and not Path("agentic-rag-turkish-docs").exists() and not Path("README.md").exists():
    !git clone -q https://github.com/alimdemir/agentic-rag-turkish-docs.git
if IN_COLAB and Path("agentic-rag-turkish-docs").exists():
    !git -C agentic-rag-turkish-docs pull -q
    %cd agentic-rag-turkish-docs
    !pip -q install -r requirements-agent.txt
elif Path.cwd().name == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
print("çalışma klasörü:", Path.cwd().name)

### Yerel model sunucusu (Ollama)

In [ ]:
!apt-get -qq install -y zstd > /dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1

import subprocess, time
ollama = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)
MODEL = "qwen3:4b-instruct-2507-q8_0"
!ollama pull {MODEL} > /dev/null 2>&1
!ollama list

### Getirme hattı (03 not defteriyle aynı)

In [ ]:
import logging, warnings
warnings.filterwarnings("ignore")
logging.getLogger("faiss").setLevel(logging.WARNING)

from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from sentence_transformers import CrossEncoder
from rag_utils import load_chunks, E5Embeddings

chunks = load_chunks(chunk_size=350, chunk_overlap=50)
vs = FAISS.from_documents(chunks, E5Embeddings(device="cuda"))
hybrid = EnsembleRetriever(retrievers=[BM25Retriever.from_documents(chunks, k=10),
                                       vs.as_retriever(search_kwargs={"k": 10})], weights=[0.5, 0.5])
reranker = CrossEncoder("BAAI/bge-reranker-v2-m3", device="cuda", max_length=512)

def search(query):
    cands = hybrid.invoke(query)[:10]
    scores = reranker.predict([(query, d.page_content) for d in cands])
    return sorted(zip(cands, map(float, scores)), key=lambda x: -x[1])

ESIK = 0.105  # 03 not defterinde yanıtı olan / olmayan soruların skorlarından seçildi
print(f"{len(chunks)} parça indekslendi, eşik {ESIK}")

### Deep Agent

In [ ]:
from langchain_ollama import ChatOllama
from deep_rag import build_agent, make_search_tool, trace_lines

model = ChatOllama(model=MODEL, temperature=0, num_ctx=16384)
agent = build_agent(model, make_search_tool(search, threshold=ESIK))

def sor(soru):
    t0 = time.time()
    r = agent.invoke({"messages": [{"role": "user", "content": soru}]}, config={"recursion_limit": 40})
    print("SORU:", soru, "\n")
    print("\n".join(trace_lines(r["messages"])))
    print(f"\nsüre: {time.time() - t0:.1f} sn · plan: {len(r.get('todos', []))} adım")
    return r

#### Çok konulu soru: planlama + iki ayrı arama

In [ ]:
r1 = sor("Yeni başlayan bir stajyer ilk hafta neler yapıyor? Ayrıca uzaktan çalışırken aday verisini kendi bilgisayarıma indirebilir miyim?")

#### Bir kısmı belgelerde olmayan soru

In [ ]:
r2 = sor("Mülakat video kayıtları kaç gün saklanıyor ve şirketin yemek kartı limiti ne kadar?")